<a href="https://colab.research.google.com/github/thomassd17/INF220-EstructuraDeDatos1/blob/main/unidad1/Conjuntos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Caso 1 — Conjunto dinámico (basado en clase, usando lista de Python)

Estructura **dinámica**: el conjunto puede crecer o achicarse libremente porque usa una `list` de Python como contenedor interno.

In [ ]:
class Conjunto:
    def __init__(self, elementos=None):
        self.elementos = []
        if elementos is not None:
            for elemento in elementos:
                self.agregar(elemento)   # se reutiliza agregar() para respetar unicidad desde el inicio

    def agregar(self, elemento):
        if elemento not in self.elementos:
            self.elementos.append(elemento)

    def quitar(self, elemento):
        if elemento in self.elementos:
            self.elementos.remove(elemento)

    def contiene(self, elemento):
        return elemento in self.elementos

    def union(self, otro):
        nuevo = Conjunto(self.elementos)
        for elemento in otro.elementos:
            nuevo.agregar(elemento)
        return nuevo

    def interseccion(self, otro):
        nuevo = Conjunto()
        for elemento in self.elementos:
            if otro.contiene(elemento):
                nuevo.agregar(elemento)
        return nuevo

    def diferencia(self, otro):
        nuevo = Conjunto()
        for elemento in self.elementos:
            if not otro.contiene(elemento):
                nuevo.agregar(elemento)
        return nuevo

    def __len__(self):
        return len(self.elementos)

    def __str__(self):
        return "{" + ", ".join(map(str, self.elementos)) + "}"


# --- Ejemplo de uso ---
conjunto_a = Conjunto([1, 2, 3])
conjunto_b = Conjunto([3, 4, 5])

print("Conjunto A:", conjunto_a)
print("Conjunto B:", conjunto_b)
print("Union:", conjunto_a.union(conjunto_b))
print("Interseccion:", conjunto_a.interseccion(conjunto_b))
print("Diferencia (A-B):", conjunto_a.diferencia(conjunto_b))
print("Tamanio de A:", len(conjunto_a))
print("A contiene el 2?:", conjunto_a.contiene(2))


Conjunto A: {1, 2, 3}
Conjunto B: {3, 4, 5}
Union: {1, 2, 3, 4, 5}
Interseccion: {3}
Diferencia (A-B): {1, 2}
Tamanio de A: 3
A contiene el 2?: True


---
## Caso 2 — Conjunto estático (vector de tamaño fijo)

Estructura **estática**: se reserva un vector de capacidad fija. Si se intenta agregar más elementos de los que caben (y no son duplicados), se controla el desbordamiento.

In [ ]:
class ConjuntoEstatico:
    def __init__(self, capacidad):
        if capacidad <= 0:
            raise ValueError("La capacidad debe ser mayor a 0")
        self.capacidad = capacidad
        self.vector = [None] * capacidad   # vector de tamanio fijo
        self.cantidad = 0                   # cuantos espacios estan realmente ocupados

    def esta_lleno(self):
        return self.cantidad == self.capacidad

    def contiene(self, elemento):
        for i in range(self.cantidad):
            if self.vector[i] == elemento:
                return True
        return False

    def agregar(self, elemento):
        if self.contiene(elemento):
            return   # ya existe, no se duplica
        if self.esta_lleno():
            raise OverflowError("El conjunto estatico esta lleno")
        self.vector[self.cantidad] = elemento
        self.cantidad += 1

    def quitar(self, elemento):
        for i in range(self.cantidad):
            if self.vector[i] == elemento:
                # se desplazan los elementos siguientes una posicion hacia atras
                for j in range(i, self.cantidad - 1):
                    self.vector[j] = self.vector[j + 1]
                self.vector[self.cantidad - 1] = None
                self.cantidad -= 1
                return
        raise ValueError(f"El elemento {elemento} no esta en el conjunto")

    def __str__(self):
        return "{" + ", ".join(map(str, self.vector[:self.cantidad])) + "}"


# --- Ejemplo de uso ---
conjunto_vector = ConjuntoEstatico(capacidad=3)
conjunto_vector.agregar(10)
conjunto_vector.agregar(20)
conjunto_vector.agregar(10)   # duplicado, se ignora

print("Conjunto estatico:", conjunto_vector)

conjunto_vector.agregar(30)
print("Conjunto estatico lleno:", conjunto_vector)

try:
    conjunto_vector.agregar(40)   # excede la capacidad
except OverflowError as e:
    print("Error controlado:", e)

conjunto_vector.quitar(20)
print("Conjunto estatico tras quitar 20:", conjunto_vector)


Conjunto estatico: {10, 20}
Conjunto estatico lleno: {10, 20, 30}
Error controlado: El conjunto estatico esta lleno
Conjunto estatico tras quitar 20: {10, 30}


---
## Caso 3 — Código con error (para corregir)

Código con dos errores: un typo en el constructor, y una operación `agregar` que no valida duplicados, rompiendo la propiedad fundamental de "elementos únicos" de un conjunto.

In [ ]:
# CODIGO CON ERROR (tal como se recibio)
class conjunto_con_error:
    def __ini__(self):                 # ERROR 1: typo, deberia ser __init__
        self.Nombre = None
        self.Elemento = []

    def agregar(self, valor):
        self.Elemento.append(valor)    # ERROR 2: no verifica si 'valor' ya existe -> permite duplicados


conjunto_prueba = conjunto_con_error()   # falla: __ini__ nunca se ejecuto


**¿Por qué falla?**
- `def __ini__(self):` está mal escrito (falta una `t`). Python nunca ejecuta ese método al crear el objeto, así que `self.Elemento` no llega a existir y cualquier uso posterior lanza `AttributeError`.
- Aunque se corrigiera el nombre, `agregar` hace `self.Elemento.append(valor)` sin verificar si el valor **ya está** en la lista, lo que permite duplicados — contradice la definición misma de un conjunto (elementos únicos).

**Corrección:**

In [ ]:
# CODIGO CORREGIDO
class ConjuntoConNombre:
    def __init__(self, nombre=None):
        self.nombre = nombre
        self.elementos = []   # renombrado a minusculas (PEP 8)

    def agregar(self, valor):
        if valor not in self.elementos:   # se valida unicidad antes de agregar
            self.elementos.append(valor)

    def __str__(self):
        return f"{self.nombre}: " + "{" + ", ".join(map(str, self.elementos)) + "}"


# --- Prueba corregida ---
conjunto_prueba = ConjuntoConNombre(nombre="Numeros de prueba")
conjunto_prueba.agregar(5)
conjunto_prueba.agregar(5)   # intento de duplicado
conjunto_prueba.agregar(7)

print(conjunto_prueba)
assert conjunto_prueba.elementos == [5, 7]
print("Correcto: no se permiten elementos duplicados")


Numeros de prueba: {5, 7}
Correcto: no se permiten elementos duplicados


---
## Caso 4 — ADT Conjunto con lista enlazada (nodos dinámicos)

Implementación **dinámica pura**, sin usar `list` de Python como contenedor: cada elemento vive en un `NodoConjunto`, enlazado al siguiente.

In [ ]:
class NodoConjunto:
    def __init__(self, valor):
        self.valor = valor
        self.siguiente = None


class ConjuntoEnlazado:
    def __init__(self):
        self.cabeza = None
        self._cantidad = 0

    def contiene(self, valor):
        actual = self.cabeza
        while actual is not None:
            if actual.valor == valor:
                return True
            actual = actual.siguiente
        return False

    def agregar(self, valor):
        if self.contiene(valor):
            return   # se respeta la unicidad
        nuevo_nodo = NodoConjunto(valor)
        nuevo_nodo.siguiente = self.cabeza
        self.cabeza = nuevo_nodo
        self._cantidad += 1

    def quitar(self, valor):
        actual = self.cabeza
        anterior = None
        while actual is not None:
            if actual.valor == valor:
                if anterior is None:
                    self.cabeza = actual.siguiente
                else:
                    anterior.siguiente = actual.siguiente
                self._cantidad -= 1
                return
            anterior = actual
            actual = actual.siguiente

    def tamanio(self):
        return self._cantidad

    def a_lista(self):
        valores = []
        actual = self.cabeza
        while actual is not None:
            valores.append(actual.valor)
            actual = actual.siguiente
        return valores

    def __str__(self):
        return "{" + ", ".join(map(str, self.a_lista())) + "}"


# --- Ejemplo de uso ---
conjunto_enlazado = ConjuntoEnlazado()
conjunto_enlazado.agregar("rojo")
conjunto_enlazado.agregar("verde")
conjunto_enlazado.agregar("rojo")   # duplicado, se ignora
conjunto_enlazado.agregar("azul")

print("Conjunto enlazado:", conjunto_enlazado)
print("Tamanio:", conjunto_enlazado.tamanio())
print("Contiene 'verde'?:", conjunto_enlazado.contiene("verde"))

conjunto_enlazado.quitar("verde")
print("Conjunto enlazado tras quitar 'verde':", conjunto_enlazado)


Conjunto enlazado: {azul, verde, rojo}
Tamanio: 3
Contiene 'verde'?: True
Conjunto enlazado tras quitar 'verde': {azul, rojo}
